In [1]:
from pyod.models.ecod import ECOD
from sklearn.preprocessing import MinMaxScaler

import pandas as pd
import numpy as np

import os
import pickle

from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt 
%matplotlib inline

import sys
sys.path.append('..')
sys.path.append('../..')

from src.utils import train_test_anomaly, raw_thresholds

In [15]:
file_path1 = '../../datasets/NAB/NAB_data_tweets_2.out'
file_path2 = '../../datasets/NAB/NAB_data_tweets_3.out'

columns = ['value', 'anomaly']

train_df = pd.read_csv(file_path1, names=columns, header=None)
test_df = pd.read_csv(file_path2, names=columns, header=None)

In [16]:
train_df['anomaly'].value_counts()

anomaly
0.0    14254
1.0     1576
Name: count, dtype: int64

In [17]:
test_df['anomaly'].value_counts()

anomaly
0.0    14311
1.0     1590
Name: count, dtype: int64

In [18]:
train_np = train_df[['value']][train_df['anomaly'] == 0]
train_np

,value
0,43.0
1,55.0
2,64.0
3,93.0
4,104.0
...,...
15825,51.0
15826,54.0
15827,46.0
15828,56.0


In [19]:
num = train_np.shape[0]
train_np = train_np.values.reshape(num, -1)
model=ECOD(contamination=0.054)
model.fit(train_np)

ECOD(contamination=0.054, n_jobs=1)

### Evaluating the Model

In [20]:
anomaly_scores = model.decision_function(test_df[['value']])
print(f'max score: {np.max(anomaly_scores)}')
print(f'min score: {np.min(anomaly_scores)}')

max score: 7.018269159880758
min score: 0.6853843205341937


In [21]:
thres = raw_thresholds(anomaly_scores, contamination=0.062)
thres

2.3327144443050174

In [31]:
thres_np = [1.0 if (score > thres) else 0 for score in anomaly_scores]
thes_np = np.array(thres_np)
np.unique(thes_np)

array([0., 1.])

In [27]:
gtruth_np = test_df[['anomaly']]
gtruth_np

,anomaly
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
15896,0.0
15897,0.0
15898,0.0
15899,0.0


In [28]:
prec = precision_score(gtruth_np, thes_np, pos_label=1)
recall = recall_score(gtruth_np, thes_np, pos_label=1)
f1 = f1_score(gtruth_np, thes_np, pos_label=1)

In [30]:
print(f'Precsion Score: {prec:.4f}  Recall: {recall:.4f}  f1_score: {f1:.4f}')

Precsion Score: 1.0000  Recall: 0.0025  f1_score: 0.0050
